In [90]:
#import for predifined Tools Creation
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

#wikipedia tool
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_char_max=250)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki, verbose=True)
wiki.name

# arxiv tool
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1, doc_content_char_max=250)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv, verbose=True)

## import for custom tool
from langchain_community.document_loaders import WebBaseLoader,PyPDFLoader,OnlinePDFLoader
from langchain_community.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.tools.retriever import create_retriever_tool

loader = PyPDFLoader("/workspaces/GenAI-Krishnaik/SearchEngineWithToolAgent/modi speech.pdf")
#loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectordb = Chroma.from_documents(documents, HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))
retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={"k":3})

from langchain.tools.retriever import create_retriever_tool
retriever_tool =create_retriever_tool(retriever, "Modi-speech","search about modi speech")

#list the all created tool
tools = [wiki, arxiv, retriever_tool]


In [91]:
print(f" 1. {wiki.name} 2. {arxiv.name} 3. {retriever_tool.name}")

 1. wikipedia 2. arxiv 3. Modi-speech


In [92]:
#Run all the toods with agents and LLM
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant",temperature=0.3)

## prompt template using 'hub'
from langchain import hub
prompt = hub.pull("hwchase17/eli5-solar-system")
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Your name is Miles. You are an astronomer who is knowledgeable about the solar system. Respond in short sentences. Shape your response as if talking to a 10-years-old.'), additional_kwargs={}),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]

In [1]:
#Creat ChatpromptTemplate
from langchain.prompts import ChatPromptTemplate,MessagesPlaceholder
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant and please do not give any summary in response. "),
    ("human", "{input}, /n Answer: "),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])
chat_prompt.messages

/workspaces/GenAI-Krishnaik/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant and please do not give any summary in response. '), additional_kwargs={}),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}, /n Answer: '), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [94]:
#crea Agent and AgentExecutor
from langchain.agents import create_openai_tools_agent
agent = create_openai_tools_agent(llm,tools, chat_prompt)

## Agent Executer
from langchain.agents import AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [95]:
agent_executor.invoke(
    {"input":"What is the paper 1706.03762 about?"}
    )




> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks in an encoder-decoder configuration. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer, based
solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to be
superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014
English-to-German translation task, improving over the existing best results,
including ensembles by over 

APIError: tool call validation failed: attempted to call tool 'brave_search' which was not in request.tools

In [89]:
agent_executor.invoke(
    {"input":"what is transformer? keep your response in short"}
    )



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': 'Transformer (machine learning)'}`


Published: 2025-02-03
Title: Aspects of Artificial Intelligence: Transforming Machine Learning Systems Naturally
Authors: Xiuzhan Guo
Summary: In this paper, we study the machine learning elements which we are interested
in together as a machine learning system, consisting of a collection of machine
learning elements and a collection of relations between the elements. The
relations we concern are algebraic operations, binary relations, and binary
relations with composition that can be reasoned categorically. A machine
learning system transformation between two systems is a map between the
systems, which preserves the relations we concern. The system transformations
given by quotient or clustering, representable functor, and Yoneda embedding
are highlighted and discussed by machine learning examples. An adjunction
between machine learning systems, a special machine learning sy

{'input': 'what is transformer? keep your response in short',
 'output': '<function='}

So let me summarize this what we use so far.
1. we used 3 tools, ArxivQueryRun, WikipediaQueryRun and Custome tool
2. 


